# False alarms at 15,000-characteristic scale

An SPC platform monitoring ~**15,000 measured characteristics** in a semiconductor assembly &
test (OSAT) production environment runs an X-bar chart on each one every shift. At the usual
**3σ limits** (α ≈ 0.0027 per chart, in control), how many **false alarms** should a shift expect
even when nothing drifted?

**Answer:** about **40.5** per shift — and the probability of *at least one* is effectively 100%.

This notebook is a thin wrapper around the committed Monte Carlo in
`../scripts/spc_multiplicity.py` (fixed seed). No production data; the scale and α match a real
platform, the alarms are synthetic.

In [ ]:
import json, pathlib, subprocess, sys
from IPython.display import SVG, display

subprocess.run([sys.executable, "../scripts/spc_multiplicity.py"], check=True)
stats = json.loads(pathlib.Path("../data/spc_multiplicity_results.json").read_text())
print(json.dumps({k: stats[k] for k in [
    "n_characteristics_production", "per_chart_alpha", "expected_false_alarms_per_shift",
    "sim_mean_unadjusted", "sim_mean_bonferroni", "sim_mean_bh", "example_shift"]}, indent=2))

## Figures

In [ ]:
FIGS = pathlib.Path("../docs/figures")
for name in ("mult-fig1.svg", "mult-fig2.svg", "mult-fig3.svg"):
    display(SVG((FIGS / name).read_text()))

## Verdict

When dozens of charts alarm on the same shift with no assignable cause, the honest unit of
inference is the **family of tests** — not each chart in isolation. Multiplicity control
(Bonferroni for strict FWER, Benjamini–Hochberg for FDR) is workload management at this scale,
not optional statistics.